# Project: Sampled vs. full-ranking metrics 🎯
Many recommender papers rank the held-out item against **99 random negatives** instead of the whole catalogue, because it's cheaper. Krichene & Rendle showed this can paint a very different picture. Measure it yourself on MovieLens with three simple models: random, popularity and SVD.

Topic: [[Evaluating Recommenders]], [[Collaborative Filtering and Matrix Factorization]] · guide note: [[Project - Sampled vs Full Ranking Metrics]] · same split as [[Project - MovieLens Recommender]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import io, pathlib, urllib.request, zipfile, numpy as np, pandas as pd
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
DATA = pathlib.Path("data/ml-latest-small")
if not DATA.exists():
    zipfile.ZipFile(io.BytesIO(urllib.request.urlopen("https://files.grouplens.org/datasets/movielens/ml-latest-small.zip").read())).extractall("data")
r = pd.read_csv(DATA / "ratings.csv"); pos = r[r.rating >= 4].sort_values(["userId", "timestamp"])
pos["rank"] = pos.groupby("userId").cumcount(ascending=False)                     # leave-last-out, as in the MovieLens project
test = pos[pos["rank"] == 0]; train = pos[pos["rank"] >= 1]
uid = {u: i for i, u in enumerate(sorted(r.userId.unique()))}; iid = {m: i for i, m in enumerate(sorted(r.movieId.unique()))}
U = csr_matrix((np.ones(len(train)), (train.userId.map(uid), train.movieId.map(iid))), shape=(len(uid), len(iid)))
seen = {u: set(U[u].indices) for u in range(U.shape[0])}
held = dict(zip(test.userId.map(uid), test.movieId.map(iid)))
pop = np.asarray(U.sum(0)).ravel()
P, s_, Qt = svds(U.astype(float), k=64); svd_scores = (P * s_) @ Qt
MODELS = {"random": lambda u: np.random.default_rng(u).random(U.shape[1]), "popularity": lambda u: pop.astype(float), "SVD k=64": lambda u: svd_scores[u]}
print(f"{U.shape[0]} users, {U.shape[1]} items, {len(held)} held-out positives")

## 1. Full-ranking hit@k
Score **every** item, remove the user's training items (`seen`), and check whether the held-out item ranks in the top `k` (rank = number of unseen items scoring strictly higher).

In [ ]:
def hit_full(scores, held_out, seen, k=10):
    # TODO 1: mask seen items, count items scoring above the held-out one, hit if < k
    raise NotImplementedError  # TODO 1

## 2. Sampled hit@k
Rank the held-out item only against `n_neg` random items the user hasn't seen (sample without replacement from all items minus `seen` minus the held-out item).

In [ ]:
def hit_sampled(scores, held_out, seen, rng, k=10, n_neg=99):
    # TODO 2: rank among n_neg sampled unseen negatives
    raise NotImplementedError  # TODO 2

In [ ]:
def _t_hits():
    sc = np.array([5, 4, 3, 2, 1.0]); rng = np.random.default_rng(0)
    return hit_full(sc, 1, {0}, k=1) and not hit_full(sc, 3, set(), k=2) and hit_sampled(sc, 0, set(), rng, k=1, n_neg=4) and not hit_sampled(sc, 4, set(), rng, k=1, n_neg=4)
check("hit@k", _t_hits, "Item 1 is first once item 0 (seen) is masked; item 3 has 3 items above it.")
if ready("hit@k"):
    rows = {}
    for name, f in MODELS.items():
        rng = np.random.default_rng(0)
        rows[name] = {"sampled HR@10": np.mean([hit_sampled(f(u), i, seen[u], rng) for u, i in held.items()]),
                      "full HR@10": np.mean([hit_full(f(u), i, seen[u]) for u, i in held.items()])}
    res = pd.DataFrame(rows).T; print(res.round(3))
    check("sampled inflates", lambda: (res["sampled HR@10"] > 3 * res["full HR@10"]).all() and abs(res.loc["random", "sampled HR@10"] - 0.10) < 0.03,
          "Sampled HR@10 should be several times higher than full HR@10, and random should score ≈ 10/100 when sampled.")
    print("→ Random gets 10 % 'for free' with 99 negatives. The gaps between models shrink or change; only full ranking reflects what a user sees.")

<details><summary>▶ Solution</summary>

```python
def hit_full(scores, held_out, seen, k=10):
    s = scores.copy()
    s[list(seen)] = -np.inf
    rank = int((s > s[held_out]).sum())
    return rank < k
```

```python
def hit_sampled(scores, held_out, seen, rng, k=10, n_neg=99):
    pool = np.setdiff1d(np.arange(len(scores)), list(seen) + [held_out])
    negs = rng.choice(pool, n_neg, replace=False)
    rank = int((scores[negs] > scores[held_out]).sum())
    return rank < k
```
</details>

## Stretch goals
- Compare NDCG@10 and recall@100 too. Do the model rankings agree across metrics?
- Plot sampled HR@10 vs. the number of negatives (9, 99, 999, all).
- Read Krichene & Rendle's correction for sampled metrics and implement it.